# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata object
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.date_published}\nVersion: {metadata.version}\n")
print(f"Cite as: {metadata.cite_as}\n")

## 2. Data Overview
Review available **record sets**, fields, and their `@id`s.

In Croissant, datasets are organized into _record sets_ (`cr:RecordSet`). Each record set contains fields/columns. We reference all entities by their `@id` value.

In [ ]:
# List available record sets
record_sets = list(dataset.record_sets)
print(f"Record Sets ({len(record_sets)} total):")
for rs in record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")

# Preview fields for each record set
for rs in record_sets:
    print(f"\nFields for Record Set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"    Field: {field.name} | @id: {field.id} | Data Type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we demonstrate iterating over records using a RecordSet's `@id`.

**Note**: Update the `selected_record_set_id` and `field @id`s as needed using the information from the previous overview.

In [ ]:
# Choose the primary record set (assume the typical name is 'Proteins' or similar)
# For this FAIR^2 dataset the main table is likely the first record set
selected_record_set = record_sets[0]  # You can change index if needed
selected_record_set_id = selected_record_set.id
print(f"Selected Record Set: {selected_record_set.name} (@id: {selected_record_set_id})")  # e.g., cr:recordSet/Proteins

# Load all records from this record set as a DataFrame
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print("Available Columns (field @ids):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter records, normalize a numeric field, and group by a categorical field. All fields are referenced by their `@id`.

> For this dataset, common numeric fields may include coverage, peptide counts, or molecular weight (`MW`). Common group-by fields include gene symbol or sample type.

In [ ]:
# Assign the desired numeric and grouping fields by their @id
# Update the following @id values based on the output in the previous cell
# For illustration, assume:
# - numeric_field_id = '@id:coverage'  (e.g., protein "coverage"%)
# - group_field_id = '@id:gene_symbol' (e.g., gene symbol)

# Replace these with actual @id values if different
numeric_field_id = df.columns[[
    'coverage' in x.lower() or 'mw' in x.lower() or 'peptide' in x.lower()
    for x in df.columns
]][0]  # choose the first matching column
group_field_id = df.columns[[ # try to find a gene symbol/group column
    'gene' in x.lower() or 'sample' in x.lower() or 'group' in x.lower()
    for x in df.columns
]][0] if any(['gene' in x.lower() or 'sample' in x.lower() or 'group' in x.lower() for x in df.columns]) else None

print(f"Numeric field selected for filtering and normalization: {numeric_field_id}")
if group_field_id:
    print(f"Grouping field selected: {group_field_id}")

# Filter: Only records with numeric_field_id > threshold
threshold = 10
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    # Attempt to coerce to float if parsing fails
    filtered_df = df.copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"\nRecords with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# If grouping field exists, group accordingly
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} per {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

Here we provide examples using matplotlib and seaborn. You can specify desired field `@id`s as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
if len(filtered_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (>{threshold})")
    plt.xlabel(numeric_field_id)
    plt.show()

# Boxplot by group if grouping field exists
if group_field_id and len(filtered_df[group_field_id].dropna().unique()) > 1:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR2 mass spectrometry record sets using their `@id`s via `mlcroissant`
- Extracted and visualized fields such as protein coverage, molecular weight, and peptide counts
- Filtered, normalized, and grouped the data for basic EDA
- Used field and record set `@id`s throughout for robust referencing

You can further extend this analysis with more feature engineering, machine learning, or hypothesis testing as required.